# 00 — Setup Verification & Dataset Overview

This notebook confirms that:
1. Environment configuration (`.env`) is loaded correctly
2. vLLM server on Euler is reachable via SSH tunnel
3. All datasets (PF exercises, GSM8K, Eedi) are loaded and well-defined

**Prerequisites:** SSH tunnel must be open (`./scripts/euler/connect.sh`)

## 1. Environment Configuration

In [2]:
import sys
sys.path.insert(0, "..")

from src.utils.config import get_settings

settings = get_settings()
print(f"vLLM Base URL:   {settings.vllm_base_url}")
print(f"Model:           {settings.vllm_model}")
print(f"Max Tokens:      {settings.vllm_max_tokens}")
print(f"Temperature:     {settings.vllm_temperature}")

vLLM Base URL:   http://localhost:8000/v1
Model:           Qwen/Qwen3-14B
Max Tokens:      2048
Temperature:     0.7


## 2. vLLM Connection Test

In [1]:
from src.utils.llm_client import LLMClient

client = LLMClient()
print(f"Server reachable: {client.is_available()}")

if client.is_available():
    resp = client.generate_text("What is 2+2? Answer briefly.", temperature=0.0)
    print(f"Model response:   {resp}")
    print(f"Token usage:      {client.usage.total_tokens} tokens")
else:
    print("\n✗ Server not reachable — run: ./scripts/euler/connect.sh")

Server reachable: False

✗ Server not reachable — run: ./scripts/euler/connect.sh


## 3. PF Exercises Dataset

In [8]:
import yaml
from pathlib import Path

pf_dir = Path("../data/pf_exercises")
pf_files = sorted(pf_dir.glob("*.yaml"))

all_exercises = []
for f in pf_files:
    with open(f) as fh:
        data = yaml.safe_load(fh)
    source = data["metadata"]["source"]
    exercises = data["exercises"]
    for ex in exercises:
        ex["_source"] = source
        ex["_file"] = f.name
    all_exercises.extend(exercises)
    print(f"  {f.name}: {len(exercises)} exercises ({source})")

print(f"\n  Total PF exercises: {len(all_exercises)}")
print(f"  Types: {dict(sorted(((t, sum(1 for e in all_exercises if e['type']==t)) for t in set(e['type'] for e in all_exercises)), key=lambda x: -x[1]))}")

  kapur_bielaczyc_2012.yaml: 6 exercises (Kapur & Bielaczyc (2012))
  kerrigan_weber_chinn_2021.yaml: 4 exercises (Kerrigan, Weber & Chinn (2021))
  loibl_rummel_2014.yaml: 4 exercises (Loibl & Rummel (2014))
  puech_macina_2025_stratl.yaml: 2 exercises (Puech, Macina, Chatain, Sachan & Kapur (2025))

  Total PF exercises: 16
  Types: {'complex_pf_problem': 6, 'well_structured': 5, 'representational_flexibility': 1, 'ill-structured': 1, 'contrasting_cases': 1, 'invention': 1, 'conceptual': 1}

✓ PF exercises loaded


In [10]:
# Show one example PF exercise
ex = next(e for e in all_exercises if e["type"] == "complex_pf_problem")
print(f"Example PF Exercise: {ex['id']}")
print(f"  Topic:  {ex['topic']}")
print(f"  Type:   {ex['type']}")
print(f"  Phase:  {ex['phase']}")
print(f"\n  Problem (first 300 chars):")
print(f"  {ex['problem'].strip()}...")
if "expected_student_rsms" in ex:
    print(f"\n  Expected RSMs: {len(ex['expected_student_rsms'])}")
    for rsm in ex["expected_student_rsms"][:3]:
        name = rsm["name"] if isinstance(rsm, dict) else rsm
        print(f"    - {name}")

Example PF Exercise: kapur2012_complex_1
  Topic:  average_speed
  Type:   complex_pf_problem
  Phase:  generation

  Problem (first 300 chars):
  It was a bright, sunny morning and the day of the Singapore Idol auditions.
Hady and Jasmine were going to audition as a team. They were practicing at
their friend Ken's house and were planning to bike to the auditions at
Singapore Expo.

After getting directions, Hady and Jasmine left Ken's house and biked
together at Jasmine's average speed of 0.15 km/min. After biking for 25
minutes, Jasmine biked over a piece of glass and her tire went flat.

Key information from dialogue:
- Ken's house to Singapore Expo: 8.75 km total (75 km/h × 7 min)
- Distance already biked: 0.15 km/min × 25 min = 3.75 km
- Remaining distance: 5 km
- Jasmine's biking speed: 0.15 km/min (150 m/min)
- Jasmine's walking speed: ~30 m/min (450m in ~14 min)
- Hady's biking speed: 0.2 km/min (200 m/min)
- Hady's walking speed: ~50 m/min (250m in 5 min)

Plan: Jasmine rides 

## 4. GSM8K Dataset (Seed problems for generation)

In [12]:
import json

gsm8k_dir = Path("../data/gsm8k")

def load_jsonl(path):
    with open(path) as f:
        return [json.loads(line) for line in f]

gsm_train = load_jsonl(gsm8k_dir / "train.jsonl")
gsm_test = load_jsonl(gsm8k_dir / "test.jsonl")

print(f"GSM8K Dataset:")
print(f"  Train: {len(gsm_train)} problems")
print(f"  Test:  {len(gsm_test)} problems")
print(f"  Fields: {list(gsm_train[0].keys())}")

# Question length statistics
q_lengths = [len(p["question"].split()) for p in gsm_train]
print(f"\n  Question length (words): min={min(q_lengths)}, median={sorted(q_lengths)[len(q_lengths)//2]}, max={max(q_lengths)}")

# Answer steps
step_counts = [p["answer"].count("\n") + 1 for p in gsm_train]
print(f"  Solution steps: min={min(step_counts)}, median={sorted(step_counts)[len(step_counts)//2]}, max={max(step_counts)}")

GSM8K Dataset:
  Train: 7473 problems
  Test:  1319 problems
  Fields: ['question', 'answer']

  Question length (words): min=9, median=42, max=183
  Solution steps: min=3, median=4, max=10


In [13]:
# Show sample GSM8K problems
print("Sample GSM8K problems:\n")
for i, p in enumerate(gsm_train[:3]):
    final_answer = p["answer"].split("####")[-1].strip()
    print(f"[{i+1}] {p['question'][:120]}...")
    print(f"    Answer: {final_answer}")
    print()

Sample GSM8K problems:

[1] Natalia sold clips to 48 of her friends in April, and then she sold half as many clips in May. How many clips did Natali...
    Answer: 72

[2] Weng earns $12 an hour for babysitting. Yesterday, she just did 50 minutes of babysitting. How much did she earn?...
    Answer: 10

[3] Betty is saving money for a new wallet which costs $100. Betty has only half of the money she needs. Her parents decided...
    Answer: 5



## 5. Eedi Dataset (Misconceptions for student simulation)

In [14]:
import pandas as pd

eedi_dir = Path("../data/eedi")

# Load misconception mapping
misconceptions = pd.read_csv(eedi_dir / "misconception_mapping.csv")
print(f"Eedi Misconceptions:")
print(f"  Total misconceptions: {len(misconceptions)}")
print(f"  Columns: {list(misconceptions.columns)}")
print(f"\n  Sample misconceptions:")
for _, row in misconceptions.sample(5, random_state=42).iterrows():
    print(f"    [{row['MisconceptionId']}] {row['MisconceptionName']}")

print("\n✓ Misconception mapping loaded")

Eedi Misconceptions:
  Total misconceptions: 2587
  Columns: ['MisconceptionId', 'MisconceptionName']

  Sample misconceptions:
    [544] When dividing a decimal by a power of 10 adds zeroes to start of the number ignoring the placement of the initial decimal point
    [2499] Thinks the interior angles of any polygon add up to 180 degrees
    [554] Finds area of one face when asked for volume
    [422] Thinks x = y is an axis
    [798] Believes all numbers in a prime factor tree are included in the list of prime factors

✓ Misconception mapping loaded


In [15]:
# Load train set (questions with misconception labels)
eedi_train = pd.read_csv(eedi_dir / "train.csv")
print(f"Eedi Questions (train):")
print(f"  Total questions: {len(eedi_train)}")
print(f"  Columns: {list(eedi_train.columns)}")

# Show distribution of misconceptions per question
misconception_cols = [c for c in eedi_train.columns if "Misconception" in c]
print(f"\n  Misconception columns: {misconception_cols}")

# Show a sample question
sample = eedi_train.iloc[0]
print(f"\n  Sample question:")
print(f"    Subject: {sample.get('SubjectName', 'N/A')}")
print(f"    Topic:   {sample.get('TopicName', 'N/A')}")
print(f"    Q:       {sample.get('QuestionText', sample.get('Question', 'N/A'))[:150]}")
print(f"    Correct: {sample.get('CorrectAnswer', 'N/A')}")
print("\n✓ Eedi training data loaded")

Eedi Questions (train):
  Total questions: 1869
  Columns: ['QuestionId', 'ConstructId', 'ConstructName', 'SubjectId', 'SubjectName', 'CorrectAnswer', 'QuestionText', 'AnswerAText', 'AnswerBText', 'AnswerCText', 'AnswerDText', 'MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']

  Misconception columns: ['MisconceptionAId', 'MisconceptionBId', 'MisconceptionCId', 'MisconceptionDId']

  Sample question:
    Subject: BIDMAS
    Topic:   N/A
    Q:       \[
3 \times 2+4-5
\]
Where do the brackets need to go to make the answer equal \( 13 \) ?
    Correct: A

✓ Eedi training data loaded


## 6. Summary — Data Inventory

| Dataset | Purpose in Pipeline | Size | Key Fields |
|---------|-------------------|------|------------|
| **PF Exercises** | Few-shot examples + evaluation reference | 15 exercises | problem, expected_rsms, criteria |
| **GSM8K** | Seed problems (well-structured → PF transform) | 7,473 train + 1,319 test | question, answer |
| **Eedi** | Misconception grounding for student simulation | ~1,600 questions + 2,500 misconceptions | question, answer choices, misconception labels |